In [38]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import trapezoid as trapz
from scipy.ndimage import gaussian_filter

In [66]:
def neckel(mu):
    p = [0.48767921486914473,
             -1.6848471461910317,
             2.355950448408068,
             -1.827014432405401,
             1.3540877312885482,
             0.31414418403067174]
    return np.polyval(p, mu) * (mu > 0)


def moffat(x, alpha=1., beta=1.):
    return (1 + (x / alpha) ** 2) ** (-beta)


def smooth(r, R, Q, epsilon=1, **kwargs):
    p = []
    p0 = trapz(moffat(np.arange(-10000,10000,0.1), **kwargs), np.arange(-10000,10000,0.1))
    for r0 in r:
        p += [trapz(Q * moffat(R - r0, **kwargs), R)]
    p = np.array(p) / p0
    q = np.interp(r, R, Q)

    return (1 - epsilon) * q + epsilon * p

In [89]:
s = np.load('q.npz')
q = np.nan_to_num(s['q'], nan=1)
r = s['r']
n = q.shape[0]

q -= np.min(q)
q /= np.median(q[:10])
q /= 1.004

In [90]:
rsun = 282.2

R = np.arange(-np.max(r), np.max(r), 0.01)
Q = neckel(np.sqrt((1 - (R / rsun) ** 2).clip(0)))
#P = moffat(R, alpha=4., beta=3.5)

sigma = 1#0.9
P = np.exp(-R ** 2 / 2 / sigma ** 2)
P /= np.sum(P)
P = np.roll(P, -len(R) // 2)

Q = np.real(np.fft.ifft(np.fft.fft(Q) * np.fft.fft(P)))

In [96]:
q_ = smooth(r, R, Q, alpha=1.9, beta=0.9, epsilon=0.22)

plt.figure(figsize=(10,8))
plt.plot(r, q)
plt.plot(r, q_)

#plt.xlim(282-50,282+50)

plt.grid(True)
plt.tight_layout()

In [95]:
t = np.where(np.abs(r - 290) < 30)
r_ = r[t]

q_ = np.interp(r, R, Q)

smin = 1000

beta = 0.9

for epsilon in np.arange(0.1, 0.35, 0.01):
    #for beta in np.arange(0.5, 1.5, 0.1):
        for alpha in np.arange(1., 3., 0.1):
            q_ = smooth(r_, R, Q, alpha=alpha, beta=beta, epsilon=epsilon)
            s = np.std(q[t] / q_ - 1)

            if s < smin:
                smin = s
                pmin = q_.copy()
                alpha_min = alpha
                beta_min = beta

                print(alpha, beta, epsilon, s)

1.0 0.9 0.1 1.3038239036380959
1.1 0.9 0.1 1.1732100133310872
1.2000000000000002 0.9 0.1 1.0624585476186543
1.3000000000000003 0.9 0.1 0.9672553146726316
1.4000000000000004 0.9 0.1 0.8844672303166273
1.5000000000000004 0.9 0.1 0.8117614183398995
1.6000000000000005 0.9 0.1 0.7473634133186766
1.7000000000000006 0.9 0.1 0.6898984371520903
1.8000000000000007 0.9 0.1 0.6382841602756476
1.9000000000000008 0.9 0.1 0.5916564078029495
2.000000000000001 0.9 0.1 0.549316544345737
2.100000000000001 0.9 0.1 0.5106934796408703
2.200000000000001 0.9 0.1 0.4753157531286701
2.300000000000001 0.9 0.1 0.44279070410412447
2.4000000000000012 0.9 0.1 0.41278871204204554
2.5000000000000013 0.9 0.1 0.3850311238881818
2.6000000000000014 0.9 0.1 0.35928090245736405
2.7000000000000015 0.9 0.1 0.3353353109294823
2.8000000000000016 0.9 0.1 0.31302014078577517
2.9000000000000017 0.9 0.1 0.2921851244371203
2.6000000000000014 0.9 0.11 0.28306020835073376
2.7000000000000015 0.9 0.11 0.2613757186639891
2.80000000000000

In [93]:
q_ = smooth(r, R, Q, epsilon=1, alpha=1.9, beta=0.9)

In [94]:
plt.figure(figsize=(10,8))
plt.plot(r, q)
plt.plot(r, q_)
#plt.plot(r, q - 0.2 * (q_ - q))

#plt.xlim(282-50,282+50)
#plt.ylim(0, 0.8)

plt.grid(True)
plt.tight_layout()

In [46]:
plt.figure(figsize=(10,8))
plt.plot(moffat(np.arange(-100,100,0.05), alpha=2.1, beta=0.9))

In [78]:
trapz(moffat(np.arange(-10000,10000,0.1), alpha=1.9, beta=0.9), np.arange(-10000,10000,0.1))

np.float64(6.985270171804679)